# Customer Churn Predictor — Google Colab Walkthrough

**Lumexa Data Scientist course**

A complete, real ML classification project: predict whether a telecom customer will
cancel ("churn") using the industry-famous IBM Telco Customer Churn dataset. This
notebook practices classification metrics, categorical encoding, class imbalance, and
model comparison end-to-end.

This notebook is fully self-contained and works with **Runtime → Run all** — no setup,
no API keys, no accounts, and no files to upload. The dataset is downloaded directly from
a public GitHub URL at runtime.

**What you'll do:**
1. Download and inspect the real IBM Telco Customer Churn dataset
2. Clean a messy string column (`TotalCharges`) and engineer new features
3. Train and compare Logistic Regression, Random Forest, and XGBoost
4. Evaluate with accuracy, precision, recall, F1, and a confusion matrix
5. Tune XGBoost with `GridSearchCV`
6. Use the best model to predict churn for new sample customers


In [1]:
# pandas, numpy, scikit-learn are preinstalled in Google Colab.
# xgboost is not preinstalled by default, so we install it quietly.
%pip install -q xgboost

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                              precision_score, recall_score)
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier

print("Libraries loaded.")


Note: you may need to restart the kernel to use updated packages.


Libraries loaded.


## Step 1: Download and load the dataset

**IBM Telco Customer Churn**

- Source URL: `https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv`
- License: IBM sample dataset, freely distributed for education/demo use.
- Rows: 7,043. Columns: 21 (customer demographics, account info, services subscribed,
  `MonthlyCharges`, `TotalCharges`, and target `Churn`).


In [2]:
DATA_URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"

df = pd.read_csv(DATA_URL)
print(f"Loaded {len(df)} rows, {len(df.columns)} columns.")
df.head()


Loaded 7043 rows, 21 columns.


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## Step 2: Inspect data quality

Real data is never perfectly clean. Let's measure the actual quality issues directly.


In [3]:
print(f"Missing values (pandas' view): {df.isna().sum().sum()}")
print(f"Duplicate rows: {df.duplicated().sum()}")

blank_total_charges = (df["TotalCharges"].astype(str).str.strip() == "").sum()
print(f"\nBlank-string rows in TotalCharges: {blank_total_charges}")
print("All of these are customers with tenure == 0:",
      (df.loc[df["TotalCharges"].astype(str).str.strip() == "", "tenure"] == 0).all())

print("\nClass balance (Churn):")
print(df["Churn"].value_counts())
churn_rate = (df["Churn"] == "Yes").mean()
print(f"Churn rate: {churn_rate:.2%}")


Missing values (pandas' view): 0
Duplicate rows: 0

Blank-string rows in TotalCharges: 11
All of these are customers with tenure == 0: True

Class balance (Churn):
Churn
No     5174
Yes    1869
Name: count, dtype: int64
Churn rate: 26.54%


We should see **0** missing values as pandas reports them, but `TotalCharges` is
stored as **text** and contains **11** rows that are blank strings (`" "`) — all of these
are brand-new customers with `tenure == 0` who haven't been billed yet. We'll coerce this
column to numeric and fill those blanks with `0.0`.

The class balance is about **26.5% churn** — a real, moderately imbalanced target, which
is why we look at precision/recall/F1, not just accuracy, when comparing models.


## Step 3: Clean the data and engineer features


In [4]:
def load_and_clean(data):
    data = data.copy()
    # TotalCharges is stored as a string and has 11 blank " " values for
    # customers with tenure == 0 (brand-new customers). Coerce to numeric and
    # fill those with 0.0 (no charges billed yet).
    data["TotalCharges"] = pd.to_numeric(data["TotalCharges"], errors="coerce")
    data["TotalCharges"] = data["TotalCharges"].fillna(0.0)

    data = data.drop(columns=["customerID"])
    data["Churn"] = data["Churn"].map({"Yes": 1, "No": 0})
    return data


def engineer_features(data):
    data = data.copy()
    data["avg_monthly_spend"] = data["TotalCharges"] / data["tenure"].replace(0, 1)
    data["tenure_years"] = data["tenure"] / 12.0
    return data


df = load_and_clean(df)
df = engineer_features(df)
print(f"Rows after cleaning: {len(df)}, columns: {len(df.columns)}")
df[["TotalCharges", "avg_monthly_spend", "tenure_years", "Churn"]].head()


Rows after cleaning: 7043, columns: 22


,TotalCharges,avg_monthly_spend,tenure_years,Churn
0,29.85,29.850000,0.083333,0
1,1889.50,55.573529,2.833333,0
2,108.15,54.075000,0.166667,1
3,1840.75,40.905556,3.750000,0
4,151.65,75.825000,0.166667,1


## Step 4: Split into train/test sets

We stratify on `Churn` so the 20% test set keeps the same churn rate as the full dataset,
which matters for a moderately imbalanced target like this one.


In [5]:
target = "Churn"
y = df[target]
X = df.drop(columns=[target])

categorical_features = [
    c for c in X.columns
    if X[c].dtype == object or pd.api.types.is_string_dtype(X[c])
]
numeric_features = [c for c in X.columns if c not in categorical_features]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train rows: {len(X_train)}, Test rows: {len(X_test)}")
print(f"Categorical features: {categorical_features}")
print(f"Numeric features: {numeric_features}")


Train rows: 5634, Test rows: 1409
Categorical features: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']
Numeric features: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', 'avg_monthly_spend', 'tenure_years']


## Step 5: Build a reusable preprocessing + model pipeline


In [6]:
def build_pipeline(numeric_feats, categorical_feats, model):
    preprocessor = ColumnTransformer([
        ("num", StandardScaler(), numeric_feats),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_feats),
    ])
    return Pipeline([("preprocess", preprocessor), ("model", model)])


def evaluate(name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred).tolist()
    print(f"[{name}] Acc={acc:.4f} Prec={prec:.4f} Rec={rec:.4f} F1={f1:.4f}")
    print(f"  Confusion matrix: {cm}")
    return {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1, "confusion_matrix": cm}

print("Helper functions ready.")


Helper functions ready.


## Step 6: Train and compare three models

We train a Logistic Regression baseline, a Random Forest, and an untuned XGBoost, and
evaluate all three on the same held-out test set.


In [7]:
results = {}
candidates = {}

# --- Logistic Regression baseline ---
log_pipeline = build_pipeline(
    numeric_features, categorical_features,
    LogisticRegression(max_iter=2000, random_state=42),
)
log_pipeline.fit(X_train, y_train)
log_pred = log_pipeline.predict(X_test)
results["logistic_regression"] = evaluate("LogisticRegression", y_test, log_pred)
candidates["logistic_regression"] = log_pipeline


[LogisticRegression] Acc=0.8077 Prec=0.6614 Rec=0.5642 F1=0.6089
  Confusion matrix: [[927, 108], [163, 211]]


In [8]:
# --- Random Forest ---
rf_pipeline = build_pipeline(
    numeric_features, categorical_features,
    RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1),
)
rf_pipeline.fit(X_train, y_train)
rf_pred = rf_pipeline.predict(X_test)
results["random_forest"] = evaluate("RandomForest", y_test, rf_pred)
candidates["random_forest"] = rf_pipeline


[RandomForest] Acc=0.7814 Prec=0.6100 Rec=0.4893 F1=0.5430
  Confusion matrix: [[918, 117], [191, 183]]


In [9]:
# --- XGBoost (untuned) ---
xgb_pipeline = build_pipeline(
    numeric_features, categorical_features,
    XGBClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        eval_metric="logloss", random_state=42, n_jobs=-1,
    ),
)
xgb_pipeline.fit(X_train, y_train)
xgb_pred = xgb_pipeline.predict(X_test)
results["xgboost"] = evaluate("XGBoost (untuned)", y_test, xgb_pred)
candidates["xgboost"] = xgb_pipeline


[XGBoost (untuned)] Acc=0.7963 Prec=0.6465 Rec=0.5134 F1=0.5723
  Confusion matrix: [[930, 105], [182, 192]]


## Step 7: Tune XGBoost with GridSearchCV

A small grid search (3-fold cross-validation, optimizing F1) to see if tuning improves on
the untuned XGBoost.


In [10]:
param_grid = {
    "model__max_depth": [3, 4, 6],
    "model__n_estimators": [200, 300],
    "model__learning_rate": [0.03, 0.1],
}
search = GridSearchCV(
    xgb_pipeline, param_grid=param_grid, cv=3, scoring="f1", n_jobs=-1
)
search.fit(X_train, y_train)
best_xgb = search.best_estimator_
best_pred = best_xgb.predict(X_test)
results["xgboost_tuned"] = evaluate("XGBoost (tuned)", y_test, best_pred)
results["xgboost_tuned"]["best_params"] = search.best_params_
candidates["xgboost_tuned"] = best_xgb
print("Best params:", search.best_params_)


[XGBoost (tuned)] Acc=0.8070 Prec=0.6759 Rec=0.5241 F1=0.5904
  Confusion matrix: [[941, 94], [178, 196]]
Best params: {'model__learning_rate': 0.03, 'model__max_depth': 3, 'model__n_estimators': 300}


## Step 8: Pick the best model

We select the model with the highest F1 score on the held-out test set. Given the ~26.5%
churn rate, F1 (which balances precision and recall) is a far more honest metric than
raw accuracy — a model that always predicts "No churn" would already score ~73.5%
accuracy while being useless.


In [11]:
best_name = max(results, key=lambda k: results[k]["f1"])
best_model = candidates[best_name]

print(f"Best model: {best_name} (F1={results[best_name]['f1']:.4f})")
print()
print(f"{'Model':<22}{'Accuracy':>10}{'Precision':>12}{'Recall':>10}{'F1':>10}")
for name, m in results.items():
    print(f"{name:<22}{m['accuracy']:>10.4f}{m['precision']:>12.4f}{m['recall']:>10.4f}{m['f1']:>10.4f}")
print(f"\nConfusion matrix for best model ({best_name}), rows=actual [No, Yes], cols=predicted [No, Yes]:")
print(results[best_name]["confusion_matrix"])


Best model: logistic_regression (F1=0.6089)

Model                   Accuracy   Precision    Recall        F1
logistic_regression       0.8077      0.6614    0.5642    0.6089
random_forest             0.7814      0.6100    0.4893    0.5430
xgboost                   0.7963      0.6465    0.5134    0.5723
xgboost_tuned             0.8070      0.6759    0.5241    0.5904

Confusion matrix for best model (logistic_regression), rows=actual [No, Yes], cols=predicted [No, Yes]:
[[927, 108], [163, 211]]


## Step 9: Predict churn for new sample customers

We reuse the exact same feature-engineering logic from training and apply the trained
pipeline to two new sample customers — the same ones used in the original
`scripts/predict.py`.


In [12]:
def predict_churn(model, customer: dict):
    row = pd.DataFrame([customer])
    row["avg_monthly_spend"] = row["TotalCharges"] / row["tenure"].replace(0, 1)
    row["tenure_years"] = row["tenure"] / 12.0
    pred = model.predict(row)[0]
    prob = model.predict_proba(row)[0, 1]
    return pred, prob


sample_customers = [
    {
        "gender": "Female", "SeniorCitizen": 0, "Partner": "Yes", "Dependents": "No",
        "tenure": 2, "PhoneService": "Yes", "MultipleLines": "No",
        "InternetService": "Fiber optic", "OnlineSecurity": "No", "OnlineBackup": "No",
        "DeviceProtection": "No", "TechSupport": "No", "StreamingTV": "No",
        "StreamingMovies": "No", "Contract": "Month-to-month", "PaperlessBilling": "Yes",
        "PaymentMethod": "Electronic check", "MonthlyCharges": 85.0, "TotalCharges": 170.0,
    },
    {
        "gender": "Male", "SeniorCitizen": 0, "Partner": "Yes", "Dependents": "Yes",
        "tenure": 60, "PhoneService": "Yes", "MultipleLines": "Yes",
        "InternetService": "DSL", "OnlineSecurity": "Yes", "OnlineBackup": "Yes",
        "DeviceProtection": "Yes", "TechSupport": "Yes", "StreamingTV": "Yes",
        "StreamingMovies": "Yes", "Contract": "Two year", "PaperlessBilling": "No",
        "PaymentMethod": "Bank transfer (automatic)", "MonthlyCharges": 65.0,
        "TotalCharges": 3900.0,
    },
]

for i, customer in enumerate(sample_customers, start=1):
    pred, prob = predict_churn(best_model, customer)
    label = "WILL CHURN" if pred == 1 else "will stay"
    print(f"Customer {i}: {label}  (churn probability = {prob:.2%})")


Customer 1: WILL CHURN  (churn probability = 64.21%)
Customer 2: will stay  (churn probability = 1.47%)


## Summary

- Cleaned a real messy string column (`TotalCharges`), fixing 11 blank-string rows for
  brand-new customers.
- Engineered `avg_monthly_spend` and `tenure_years` features.
- Trained and compared Logistic Regression, Random Forest, and XGBoost.
- Evaluated with accuracy, precision, recall, F1, and a confusion matrix — and discussed
  *why* F1 (not accuracy) is the right metric on this ~26.5%-imbalanced target.
- Tuned XGBoost with `GridSearchCV`.
- Used the best model to predict churn probability for new sample customers.

### Extension ideas
- Try `class_weight="balanced"` or SMOTE to address the imbalance.
- Add a feature for "number of add-on services subscribed".
- Plot a precision-recall curve instead of relying on a single classification threshold.
